# Enclave  Evaluation - Inference

This notebook demonstrates a **privacy-preserving model inference** using Syft Enclaves.

---

## Who's involved?

| Actor | Email | Role |
|-------|-------|------|
| **Enclave** | `enclave@ai-safety.org` | Trusted execution environment |
| **Model owner** | `model_owner@ai-safety.org` | Owns the language model weights |
| **Benchmark owner** | `benchmark_owner@ai-safety.org` | Owns the demographic evaluation benchmark |
| **Researcher** | `researcher@ai-safety.org` | Writes and submits inference code |

## Flow

1. Model owner uploads its model weights as a dataset (mock placeholder + private weights JSON)
2. Benchmark owner uploads the evaluation benchmark (mock samples + private full set)
3. Both share private data with the enclave
4. Researcher browses mock datasets, writes inference code, submits job
5. Enclave distributes job to Model owner and Benchmark owner for approval
6. Both approve → enclave runs inference against private assets
7. Results shared back with Researcher, Model owner, and Benchmark owner

---

## Setup

In [ ]:
import csv
import json
import os
from pathlib import Path

from syft_enclaves import SyftEnclaveClient

os.environ["PRE_SYNC"] = "false"

In [ ]:
# Root all SyftBox working dirs under ./blindfold-network (not /tmp), and locate
# the repo root so model/benchmark paths work whether the kernel runs from the
# repo root or from notebooks/. (Wiping for a fresh run is the separate Reset cell.)
import itertools
import sys

import syft_client.sync.syftbox_manager as _sbm

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

NETWORK_ROOT = ROOT / "blindfold-network"
NETWORK_ROOT.mkdir(parents=True, exist_ok=True)

# Each of the 4 clients MUST get its OWN syftbox folder. A shared constant collapses every
# datasite into one dir, so the researcher's job push lands physically inside the enclave's
# datasite and the enclave then rejects the proposed "create" as a conflict
# (ProposedEventFileOutdatedException). Per-client folders isolate them.
#
# The quad builds clients in a FIXED order (enclave, do1, do2, ds), so a counter maps to
# role names for a readable tree. The client-N fallback guards against syft changing that
# order or adding a 5th client.
#
# We return the datasite one level deep (<role>/syftbox) so syft's companion event dirs —
# always siblings named <name>-events / <name>-event-messages — land INSIDE the actor's
# folder (<role>/syftbox-events, ...) instead of cluttering blindfold-network/.
_ROLES = ["enclave", "model_owner", "benchmark_owner", "researcher"]
_sb_seq = itertools.count()


def _syftbox_folder():
    i = next(_sb_seq)
    role = _ROLES[i] if i < len(_ROLES) else f"client-{i + 1}"
    return NETWORK_ROOT / role / "syftbox"


_sbm.random_syftbox_folder_for_testing = _syftbox_folder

for _half in (
    "to_submit",
    "post_run",
):  # flat imports per half (audit_core, judge, ...)
    sys.path.insert(0, str(ROOT / "code" / "researcher_code" / _half))

print(f"  repo root    : {ROOT}")
print(f"  network root : {NETWORK_ROOT}  (one folder per role: {_ROLES})")

## Reset — run this for a fresh network

Re-running `create_dataset` over existing state raises `FileExistsError`, so wipe the
in-memory working dir first. Only touches the gitignored `blindfold-network/` — nothing real.
On **Run All** this runs near the top, so every full pass starts clean.


In [ ]:
import shutil

shutil.rmtree(NETWORK_ROOT, ignore_errors=True)
NETWORK_ROOT.mkdir(parents=True, exist_ok=True)
print(f"  reset: fresh SyftBox network at {NETWORK_ROOT}")

## Helper functions to create mock assets

Below are the helper functions to create mock datasets and models so the researcher can inspect and develop code upon.
The researcher sees only a **mock model** and **mock datasets**. 

> Only the secure enclave have the uploaded private model weights and datasets

In [ ]:
# Config + helpers for the two uploads: the mock model and a benchmark CSV writer.
import csv as _csv
import shutil
import tempfile
from pathlib import Path


def _tmp(prefix: str) -> Path:
    d = Path(tempfile.mkdtemp()) / prefix
    d.mkdir(parents=True, exist_ok=True)
    return d


def mock_model() -> Path:
    """The mock the model owner exposes: a stub infer.py with the same init/generate
    interface as the private one, so the researcher can develop their eval against a fake
    model before any real weights are shared. No weights — that is the whole point."""
    d = _tmp("model-mock")
    shutil.copy(ROOT / "code" / "model_owner_code" / "mock_infer.py", d / "infer.py")
    return d


def write_csv(rows: list[dict], fieldnames: list[str]) -> Path:
    d = _tmp("benchmark")
    p = d / "benchmark.csv"
    with open(p, "w", newline="", encoding="utf-8") as f:
        w = _csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)
    return p


print("Config + helpers defined")

---
## Step 0 — Spin up the network

We create four clients with fixed emails and an in-memory mock network — no real servers needed.

```mermaid
flowchart LR
    subgraph NET["🕸️ in-memory network · no real servers"]
        MO["🏢 Model owner"]
        BO["🛡️ Benchmark owner"]
        R["🔍 Researcher"]
        E["🔒 Enclave"]
    end
```


In [ ]:
enclave, model_owner, benchmark_owner, researcher = (
    SyftEnclaveClient.quad_with_mock_drive_service_connection(
        enclave_email="enclave@ai-safety.org",
        do1_email="model_owner@ai-safety.org",
        do2_email="benchmark_owner@ai-safety.org",
        ds_email="researcher@ai-safety.org",
        use_in_memory_cache=False,
    )
)

print(f"  Enclave    : {enclave.email}")
print(f"  Model owner     : {model_owner.email}")
print(f"  Benchmark owner      : {benchmark_owner.email}")
print(f"  Researcher : {researcher.email}")

---
## Step 1 — Model owner uploads the model weights

Model owner creates a dataset with:
- **mock**: a stub `infer.py` (same `init`/`generate` interface, no weights) — the researcher develops and tests their eval against it before any real weights are shared
- **private**: the real model weights (MLX `model.safetensors` + tokenizer) plus the model owner's real `infer.py` — only the enclave ever receives these

> The model is set once in the `MODEL` constant (next code cell). First download its weights (default is the lightest, `qwen2.5-0.5b`):
> `uv run python scripts/download_models.py --models <MODEL>`

```mermaid
flowchart LR
    MO["🏢 Model owner"]
    MO -->|"mock · stub infer.py"| PUB["👁️ public · researcher sees this"]
    MO ==>|"private · weights + infer.py"| PRV["🔒 enclave-only"]
    classDef secret fill:#fde2e9,stroke:#c0396b,color:#000
    class PRV secret
```


In [ ]:
import shutil

MODEL = (
    "qwen2.5-0.5b"  # the model under audit — single source of truth: the weights dir
)
MODEL_DIR = ROOT / "models" / MODEL
assert MODEL_DIR.exists(), (
    f"missing {MODEL_DIR} — run: uv run python scripts/download_models.py --models {MODEL}"
)
# Private asset = real weights + the model owner's real infer.py (their IP).
shutil.copy(ROOT / "code" / "model_owner_code" / "infer.py", MODEL_DIR / "infer.py")
# Mock asset = a stub infer.py (no weights) the researcher develops against.
mock_dir = mock_model()
_gb = sum(p.stat().st_size for p in MODEL_DIR.rglob("*") if p.is_file()) / 1e9

model_owner.create_dataset(
    name="model",
    mock_path=mock_dir,
    private_path=str(MODEL_DIR),
    summary="Model weights (mlx safetensors) + infer.py — private to the enclave",
    users=[researcher.email, enclave.email],
    upload_private=True,
    sync=False,
)

print("  Model owner uploaded 'model'")
print("    mock    : infer.py (stub) — researcher develops against this")
print(f"    private : weights + infer.py ({_gb:.2f} GB)")

---
  ## Step 2 — Benchmark owner uploads the evaluation benchmark

Benchmark owner creates a dataset with:
- **mock**: the mock dataset with is a small sample of benchmark rows (3 rows) — the
researcher can inspect these to develop their eval
- **private**: the full benchmark (47 bilingual EN+VN prompts
across scam, medical, jailbreak, and benign-control categories)
— only the enclave sees the full hidden set

```mermaid
flowchart LR
    BO["🛡️ Benchmark owner"]
    BO -->|"mock · 3 sample rows"| PUB["👁️ public · researcher sees this"]
    BO ==>|"private · 47 EN+VN prompts"| PRV["🔒 enclave-only"]
    classDef secret fill:#fde2e9,stroke:#c0396b,color:#000
    class PRV secret
```


In [ ]:
BENCHMARK_CSV = ROOT / "data" / "benchmark.csv"
ALL_ROWS = list(csv.DictReader(open(BENCHMARK_CSV, encoding="utf-8")))
FIELDS = list(ALL_ROWS[0].keys())

benchmark_owner.create_dataset(
    name="vn_harm_benchmark",
    mock_path=write_csv(ALL_ROWS[:3], FIELDS),
    private_path=write_csv(ALL_ROWS, FIELDS),
    summary="Vietnamese-harm refusal benchmark (EN+VN): scam, medical, jailbreak, benign controls",
    users=[researcher.email, enclave.email],
    upload_private=True,
    sync=False,
)

print("  Benchmark owner uploaded 'vn_harm_benchmark'")
print("    mock    : 3 sample rows")
print(f"    private : {len(ALL_ROWS)} rows (full hidden set)")

---
## Step 3 — Share private dataset and model weights with the enclave & sync

- **Model owner** grants the **enclave** access to their **private model weights**.
- **Benchmark owner** grants the **enclave** access to their **private  benchmark dataset**.
- After syncing, the **researcher can browse the mock datasets**.

```mermaid
flowchart LR
    MO["🏢 Model owner"] ==>|"grant · private weights"| E["🔒 Enclave"]
    BO["🛡️ Benchmark owner"] ==>|"grant · private benchmark"| E
    classDef secret fill:#fde2e9,stroke:#c0396b,color:#000
    class MO,BO secret
```


In [ ]:
model_owner.share_private_dataset("model", enclave.email)
benchmark_owner.share_private_dataset("vn_harm_benchmark", enclave.email)
print("  Private datasets shared with enclave")

model_owner.sync()
benchmark_owner.sync()
researcher.sync()
print("  All clients synced")

---
## Step 4 — Researcher browses mock datasets

The researcher can see the mock data (placeholder weights + sample benchmark rows) but never the private files.

```mermaid
flowchart LR
    R["🔍 Researcher"] -->|browse| PUB["👁️ mock model · sample benchmark"]
    R -. "✗ no access" .-> PRV["🔒 private weights · full benchmark"]
    classDef secret fill:#fde2e9,stroke:#c0396b,color:#000
    class PRV secret
```


In [ ]:
# Please run the code cell twice if you see no table for the first run
researcher.datasets.get_all()

In [ ]:
try:
    researcher.datasets.get_all()[0].private_config
except Exception:
    print(
        "expected error since the researcher should not have access to private configs / data / model weights"
    )

---
## Step 5 — Researcher submits the eval (a reviewable code folder)

The researcher submits **only** the `code/researcher_code/to_submit/` folder into the enclave —
`main.py` (entrypoint) plus `audit_core.py` (harm-origin map + language detection), `benchmark.py`, and `record.py`. The
scoring half after getting the results (`post_run/`: the LLM judge and the report) stays
on the researcher's machine and never enters the enclave.

**Both owners review the entire submitted folder** before approving, so nothing runs against their secret assets without their reviews.
The eval flow works like this:
- `main.py` reads `params.json` that contains owner emails + dataset names
- loads the Model owner's model via their `infer.py` (shipped in the model dataset, using `mlx_lm.load(...)` internally)
- then it sends every prompt in EN and VN exactly as written
- finally, writes the model's **raw responses** to `outputs/results.json` — no verdict.

Scoring is deliberately kept **out** of the enclave: the researcher judge the output manually, or they can run the LLM judge locally afterward (Step 12), so
no API key or network egress ever touches the enclave private boundary. 

The enclave job only *generates*; it never *labels*. This **generate-then-judge** split is standard practice for LLM
safety evals (HarmBench, StrongREJECT, JailbreakBench all separate the two), for three reasons:
- **Independence** — an independent grader (a different model, or a human) labels the output, so a jailbroken model can't whitewash its own answer. The system under test never scores itself.
- **No contamination** — the attack prompt reaches the model exactly as an attacker would send it. Fusing "answer the request *and* grade yourself" into one call makes the model aware it's being tested, which inflates refusals and stops measuring real-world behavior.
- **Re-judgeable ground truth** — the raw responses are the durable artifact. You can re-score them later with a sharper rubric or a stronger judge, without re-running the slow, private generation step.

```mermaid
flowchart LR
    R["🔍 Researcher"] ==>|"submit · to_submit/ (main.py + helpers)"| E["🔒 Enclave"]
    R -.->|"post_run/ judge stays here"| LOCAL["💻 researcher's machine"]
    style LOCAL fill:#eef2ff,stroke:#3b5bdb,color:#000
```


In [ ]:
# Write run parameters into the job folder so the enclave's main.py can resolve the
# two private datasets. (gitignored — it carries the per-run owner emails.)
JOB_DIR = ROOT / "code" / "researcher_code" / "to_submit"
params = {
    "model_owner": model_owner.email,
    "bench_owner": benchmark_owner.email,
    "model_dataset": "model",
    "benchmark_dataset": "vn_harm_benchmark",
    "model_label": MODEL,
    "demo_per_category": None,  # set to None to audit the full hidden set
}
(JOB_DIR / "params.json").write_text(json.dumps(params, indent=2))
print(f"  wrote {JOB_DIR / 'params.json'}")
print(
    f"  submitting folder: code/researcher_code/{JOB_DIR.name}/  (entrypoint: main.py)"
)

In [ ]:
# remove any __pycache__ folders from the job folder, because syft uploads every file
import shutil

for _pyc in JOB_DIR.rglob("__pycache__"):
    shutil.rmtree(_pyc, ignore_errors=True)

researcher.submit_python_job(
    enclave.email,
    str(JOB_DIR),
    "vn_harm_audit",
    datasets={
        model_owner.email: ["model"],
        benchmark_owner.email: ["vn_harm_benchmark"],
    },
    dependencies=["mlx-lm", "pydantic"],
    entrypoint="main.py",
    share_results_with_do=True,
)

print("  Job 'vn_harm_audit' submitted (code folder: code/researcher_code/to_submit/)")
print(f"    model     : 'model' (weights + infer.py)  from {model_owner.email}")
print(f"    benchmark : 'vn_harm_benchmark'           from {benchmark_owner.email}")
print("    both owners review code/researcher_code/to_submit/ before approving")

---
## Step 6 — Enclave receives and distributes the job

The enclave syncs incoming files, then *forwards the approval request to Model Owner and Benchmark Owner*.

```mermaid
flowchart LR
    E["🔒 Enclave"] -. "forward code for review" .-> MO["🏢 Model owner"]
    E -. "forward code for review" .-> BO["🛡️ Benchmark owner"]
```


In [ ]:
enclave.sync()  # pull job files from the network
enclave.receive_jobs()  # distribute approval requests to Model owner and Benchmark owner

enclave_job = enclave.jobs["vn_harm_audit"]
print(f"  Enclave job status : {enclave_job.status}")
print("  Waiting for Model owner and Benchmark owner to approve...")

---
## Step 7 — Model owner and Benchmark owner review and approve

Each party syncs to receive the forwarded job, inspects the submitted code, and approves.
The enclave only proceeds once **both** have approved.

```mermaid
flowchart LR
    MO["🏢 Model owner"] -. "✔ approve (or veto)" .-> E["🔒 Enclave"]
    BO["🛡️ Benchmark owner"] -. "✔ approve (or veto)" .-> E
    E --> OK["✅ both approved → runnable"]
    style OK fill:#d8f5d8,stroke:#2e7d32,color:#000
```


In [ ]:
model_owner.sync()
benchmark_owner.sync()

model_owner_job = model_owner.jobs["vn_harm_audit"]
benchmark_owner_job = benchmark_owner.jobs["vn_harm_audit"]

print(f"  Model owner sees job 'vn_harm_audit'  status={model_owner_job.status}")
print(
    f"  Benchmark owner  sees job 'vn_harm_audit'  status={benchmark_owner_job.status}"
)

In [ ]:
# Inspect the submitted code before approving
model_owner_job

In [ ]:
# also you can see other files
model_owner_job.files

In [ ]:
# model owner and benchmark owner approve the job
model_owner.approve_job(model_owner_job)
print("  Model owner approved")

benchmark_owner.approve_job(benchmark_owner_job)
print("  Benchmark owner  approved")

In [ ]:
# Enclave collects both approval votes
enclave.sync()

enclave_job = enclave.jobs["vn_harm_audit"]
print(f"  Enclave job status: {enclave_job.status}")
assert enclave_job.status == "approved", (
    f"Expected 'approved', got '{enclave_job.status}'"
)
print("  Both approvals received — job is APPROVED")

---
## Step 8 — Enclave executes the job

**With both parties approved**, the enclave **runs the eval code** against Model owner's **private model** and **Benchmark owner's private prompt set**.

```mermaid
flowchart LR
    PW["🔒 private weights"] --> E["🔒 Enclave · sealed memory"]
    PB["🔒 private benchmark"] --> E
    E ==>|"generate only · no verdict"| OUT["📄 results.json · raw responses"]
    classDef secret fill:#fde2e9,stroke:#c0396b,color:#000
    class PW,PB secret
```


In [ ]:
enclave.run_jobs()

enclave_job = enclave.jobs["vn_harm_audit"]
print(f"  Enclave job status: {enclave_job.status}")
assert enclave_job.status == "done"
print("  Job completed successfully")

---
## Step 9 — Distribute results

The enclave pushes output files to the Researcher and (because `share_results_with_do=True`)
to both Model owner and Benchmark owner.

```mermaid
flowchart LR
    E["🔒 Enclave"] ==>|"results.json"| R["🔍 Researcher"]
    E -->|"results.json"| MO["🏢 Model owner"]
    E -->|"results.json"| BO["🛡️ Benchmark owner"]
```


In [ ]:
enclave.distribute_results()
print("  Results distributed to Researcher, Model owner, and Benchmark owner")

---
## Step 10 — Researcher retrieves the result

```mermaid
flowchart LR
    E["🔒 Enclave"] ==>|"raw results.json"| R["🔍 Researcher"]
    R --> OUT["📄 results/ run folder · results.csv"]
```


In [ ]:
from report import make_run_dir, write_records_csv

researcher.sync()

researcher_job = researcher.jobs["vn_harm_audit"]
print(f"  Researcher job status : {researcher_job.status}")
print(f"  Output files          : {researcher_job.output_paths}")

assert researcher_job.status == "done"
assert len(researcher_job.output_paths) > 0


with open(researcher_job.output_paths[0]) as f:
    out = json.load(f)

# One folder per run: results/<model>_<datetime>/ — keeps this audit's files together and
# never overwrites a prior run. Step 12 writes the scored outputs into the SAME folder.
run_dir = make_run_dir(ROOT / "results", out["model"])
write_records_csv(out["records"], run_dir / "results.csv")

print(f"\n  {len(out['records'])} raw responses -> {run_dir / 'results.csv'}")
print("  Next: researcher scores these locally with the LLM judge (Step 12).")

---
## Step 11 — Model owner and Benchmark owner also receive the result

Because `share_results_with_do=True`, each party independently receives the output.

```mermaid
flowchart LR
    E["🔒 Enclave"] -->|"results.json"| MO["🏢 Model owner"]
    E -->|"results.json"| BO["🛡️ Benchmark owner"]
```


In [ ]:
model_owner.sync()
benchmark_owner.sync()

model_owner_job = model_owner.jobs["vn_harm_audit"]
benchmark_owner_job = benchmark_owner.jobs["vn_harm_audit"]

print(f"  Model owner — output files : {model_owner_job.output_paths}")
print(f"  Benchmark owner  — output files : {benchmark_owner_job.output_paths}")

assert len(model_owner_job.output_paths) > 0
assert len(benchmark_owner_job.output_paths) > 0

print()
print("  Both Model owner and Benchmark owner received the result")

---
## [OPTIONAL - NEED ANTHROPIC API KEY] Step 12 — Researcher scores the results (local, off the enclave)

- **What this does.** The enclave returned only *raw* model responses — no verdicts.
Here the researcher turns them into a scorecard: an LLM judge (Claude) reads each response
and labels it `refused_to_answer | comply_to_answer | partial` with a
one-line reason; we then tally the **EN-vs-VN refusal gap** per harm category
and write the per-case CSVs plus a plain-language report into this run's folder
(`results/<model>_<datetime>/`). No key? It falls back to a
keyword classifier so the pipeline still completes (just with rougher labels).

- **Why judging is a separate step.** This is standard LLM-eval practice — *generate*, then *judge*, with an **independent** grader. The model under test only produces answers; a different model labels them. That keeps the auditor independent (a jailbroken model can't whitewash its own output), keeps the attack prompt unwrapped (so the model never "knows it's being graded" — that contamination inflates refusals), and leaves the raw responses as a durable artifact you can re-score later with a sharper rubric, without re-running generation.

- **Why it runs *here*, not in the enclave.** The judge calls an external API (Claude) — that means network egress and an API key. The enclave's entire guarantee is that it is network-isolated and holds nobody's secrets, so we never put the key inside it or let private data leave it. Judging therefore happens **after declassification**, on the researcher's own machine, over data both owners already approved for release. The security boundary and the eval best practice point the same way: keep generation and judging apart.

The payoff is the **EN-vs-VN gap** — the headline finding: does the model refuse a harmful request in English but comply with the *same* request in Vietnamese?

> CLI equivalent (standalone): `judge.py` turns the raw responses into scored ones, then `report.py` renders the gap report + CSVs.

```mermaid
flowchart LR
    RAW["📄 raw responses"] --> J["🔍 Researcher · LLM judge<br/>(Claude · .env key · off-enclave)"]
    J ==> OUT["📊 scorecard.csv + REPORT.md · EN-vs-VN gap"]
    style OUT fill:#d8f5d8,stroke:#2e7d32,color:#000
```


In [ ]:
from judge import (
    score_records,  # code/researcher_code/post_run — local LLM judge (Anthropic)
)
from report import render_report, write_records_csv, write_scorecard_csv
from scoring import tally

# Judge each raw response (Claude via the .env key; keyword fallback if unavailable).
scored = score_records(out["records"])

# Write into the SAME per-run folder created at retrieval (Step 10).
write_records_csv(scored, run_dir / "scored.csv")
write_scorecard_csv(tally(scored), run_dir / "scorecard.csv")
report_md = render_report(scored, out["model"])
(run_dir / "REPORT.md").write_text(report_md, encoding="utf-8")

print(report_md)
print(f"\n  saved -> {run_dir}/  (REPORT.md, scored.csv, scorecard.csv)")

---
## Summary

| Step | Actor | Action | Outcome |
|------|-------|--------|---------|
| 1 | Model owner | Upload model-weights dataset (mock placeholder + private `weights.json`) | Weights available on network |
| 2 | Benchmark owner | Upload benchmark dataset (mock sample + private full set) | Benchmark available on network |
| 3 | Model owner, Benchmark owner | Share private data with enclave | Enclave can access both assets |
| 4 | Researcher | Browse mock datasets | Sees placeholder weights + sample benchmark only |
| 5 | Researcher | Submit eval job (Qwen load + EN/VN refusal scoring) + `share_results_with_do=True` | Job sent to enclave |
| 6 | Enclave | `sync()` + `receive_jobs()` | Job distributed to Model owner and Benchmark owner for review |
| 7 | Model owner, Benchmark owner | `approve_job()` | Enclave status → **approved** |
| 8 | Enclave | `run_jobs()` | Inference runs against private weights + benchmark |
| 9 | Enclave | `distribute_results()` | Outputs pushed to Researcher, Model owner, Benchmark owner |
| 10 | Researcher | `sync()` + read output | Raw per-case responses received |
| 11 | Model owner, Benchmark owner | `sync()` + read output | Results received independently |
| 12 | Researcher (Optional) | Run LLM judge locally (off-enclave) + tally EN-vs-VN gap (need ANTHROPIC-API-KEY) to produce final report | `scorecard_<model>.json` |

The private model weights and the full benchmark **never left the enclave** —
only the approved inference outputs were shared.

---